# Stability-constrained economic proportioning, *variant 2*

*Last updated 14 June 2026*

The task is
to determine $b_1$, $b_2$, $b_3$, $b_4$, $h_{\text{s}}$, and $h_{\text{c}}$,
such that the cross-sectional area $A$ is minimized,
while keeping the safety factors
against overturning, sliding, and bearing capacity failures
($s_{\text{o}}$, $s_{\text{s}}$, $s_{\text{b}}$, respectively)
within prescribed values.
Here, cross-sectional area is used as a (rather rough) economic measure of desirability:
an increase in area corresponds to an increase in material and/or labour costs.
Realize that the cross-sectional area and safety factors are dependent on the wall proportions.

## Modelling

To formalize,
- $\boldsymbol{p} \triangleq \left[b_1, b_2, b_3, b_4, h_{\text{c}}, h_{\text{s}}\right]$
  are the *optimization (or decision) variables*,
  and
- $A\!\left(\boldsymbol{p}\right)$ is the *objective function*.

For physical feasibility,
each dimension making up the wall proportions
can only be within a range of (nonnegative) allowable values.
This is expressed by the following *bounds*.
- $b_{1, \text{min}} \leq b_1 \leq b_{1, \text{max}}$
- $b_{2, \text{min}} \leq b_2 \leq b_{2, \text{max}}$
- $b_{3, \text{min}} \leq b_3 \leq b_{3, \text{max}}$
- $b_{4, \text{min}} \leq b_4 \leq b_{4, \text{max}}$
- $h_{\text{c}, \text{min}} \leq h_{\text{c}} \leq h_{\text{c}, \text{max}}$

Assuming that the heel-side face of the stem is vertical,
the base thickness $b_2$ of the stem cannot be less than its tip thickness $b_4$.
This is expressed as the *inequality constraint*
$b_2 \geq b_4$
(or,
$\left[0, 1, 0, -1, 0, 0\right]^{\mathsf{T}} \boldsymbol{p} \geq 0$
in vector form).
Realize that the "effective" lower limit on $b_2$ is
whichever is greater between $b_{2, \text{min}}$ and $b_4$,
and so the user may, for convenience, set $b_{2, \text{min}}$ to zero.

The stem height $h_{\text{s}}$ and the footing thickness $h_{\text{c}}$
must add up to the desired retained height $H$ of the backfill.
This requirement is embodied by the *equality constraint*
$h_{\text{s}} + h_{\text{c}} = H$
(or,
$\left[0, 0, 0, 0, 1, 1\right]^{\mathsf{T}} \boldsymbol{p} = H$
in vector form).
An equality constraint can be equivalently expressed as a "tight sandwiching" of two inequality constraints;
and so in this case,
$H \leq h_{\text{s}} + h_{\text{c}} \leq H$
(or,
$H \leq \left[0, 0, 0, 0, 1, 1\right]^{\mathsf{T}} \boldsymbol{p} \leq H$
in vector form).

Lastly,
the stability consideration represented by the safety factors having to take on minimum values
are expressed as *inequality constraints*.
- $s_{\text{o}} \!\left(\boldsymbol{p}; \mathcal{D}\right) \geq s_{\text{o}, \text{min}}$,
- $s_{\text{s}} \!\left(\boldsymbol{p}; \mathcal{D}\right) \geq s_{\text{s}, \text{min}}$,
  and
- $s_{\text{b}} \!\left(\boldsymbol{p}; \mathcal{D}\right) \geq s_{\text{b}, \text{min}}$.

The desired retained height,
the variable bounds,
and
the minimum required safety factors
form part of the *problem data* $\mathcal{D}$,
*i.e.*,
"input information" provided prior to optimization.
As such,
$\mathcal{D}$ also comprises parameters pertaining to
- the wall
  (*e.g.*,
  material unit weight $\gamma_{\text{w}}$,
  sometimes approximated as that of concrete);
- the soil
  (*e.g.*,
  material unit weight $\gamma_{\text{h}}$,
  internal friction angle $\phi_{\text{h}}$,
  cohesion $c_{\text{h}}$,
  and
  slope $\theta$
  of the backfill,
  and
  bearing capacity $q_{\text{a}}$
  of the soil beneath the wall);
  and
- the soil-wall interactions
  (*e.g.*,
  friction angle $\phi_{\text{b}}$ and adhesion $a_{\text{b}}$
  between the wall material and the soil beneath the wall).

The above notions constitute an *optimization model*,
expressed symbolically as

$$
\operatorname*{minimize}_{\boldsymbol{p}}
A \!\left(\boldsymbol{p}\right)
\hphantom{space}
\\
\text{subject to:} \hphantom{spacespac}
\\
\hphantom{spacespacespacespacespacespa}
b_{1, \text{min}} \leq b_1 \leq b_{1, \text{max}},
\quad
b_{2, \text{min}} \leq b_2 \leq b_{2, \text{max}},
\\
\hphantom{spacespacespacespacespacespa}
b_{3, \text{min}} \leq b_3 \leq b_{3, \text{max}},
\quad
b_{4, \text{min}} \leq b_4 \leq b_{4, \text{max}},
\\
\hphantom{spacespac}
h_{\text{c}, \text{min}} \leq h_{\text{c}} \leq h_{\text{c}, \text{max}},
\\
\hphantom{spacespacespacespacespa}
b_2 \geq b_4,
\quad \quad \quad \quad \quad \quad
h_{\text{s}} + h_{\text{c}} = H,
\\
\hphantom{spaces}
s_{\text{o}} \!\left(\boldsymbol{p}; \mathcal{D}\right) \geq s_{\text{o}, \text{min}},
\\
\hphantom{spaces}
s_{\text{s}} \!\left(\boldsymbol{p}; \mathcal{D}\right) \geq s_{\text{s}, \text{min}},
\\
\hphantom{spaces}
s_{\text{b}} \!\left(\boldsymbol{p}; \mathcal{D}\right) \geq s_{\text{b}, \text{min}}.
$$


## Utilities

In [1]:
import math as mt

import numpy as np
import scipy.optimize as spo

from numpy.typing import NDArray

In [2]:
def A(
    p : NDArray,            # Wall proportions
    D : dict[str, float],   # Problem data
) -> float:
    phi_ = mt.radians(D["phi_h"]); theta_ = mt.radians(D["theta"])
    b = p[0] + p[1] + p[2]; H = p[-2] + p[-1]

    A1 = (p[1] - p[3]) * p[-2] * 0.5
    A2 = p[3] * p[-2]
    A3 = b * p[-1]

    return A1 + A2 + A3

In [3]:
def s_osb(
    p : NDArray,            # Wall proportions
    D : dict[str, float],   # Problem data
):
    phi_h_ = mt.radians(D["phi_h"]); theta_ = mt.radians(D["theta"])
    b = p[0] + p[1] + p[2]; H = p[-2] + p[-1]; h = H + (p[2] * mt.tan(theta_))

    # Developed weight (i.e., resultant gravity load)
    A1 = (p[1] - p[3]) * p[-2] * 0.5
    A2 = p[3] * p[-2]
    A3 = b * p[-1]
    A4 = p[2] * p[-2]
    A5 = 0.5 * (p[2] ** 2) * mt.tan(theta_)
    # 
    W1 = A1 * D["y_w"]; W2 = A2 * D["y_w"]; W3 = A3 * D["y_w"]
    W4 = A4 * D["y_h"]; W5 = A5 * D["y_h"]
    # 
    x1 = p[0] + ((2 / 3) * (p[1] - p[3]))
    x2 = p[0] + p[1] - (p[3] / 2)
    x3 = b / 2
    x4 = p[0] + p[1] + (p[2] / 2)
    x5 = p[0] + p[1] + ((2 / 3) * p[2])

    # Rankine active force
    Ka = (1 - mt.sin(phi_h_)) / (1 + mt.sin(phi_h_))
    Pa = Ka * D["y_h"] * h
    Fa = 0.5 * Pa * h
    Fav = Fa * mt.sin(theta_)
    Fah = Fa * mt.cos(theta_)

    # Righting and overturning moments
    RM = (W1 * x1) + (W2 * x2) + (W3 * x3) + (W4 * x4) + (W5 * x5) + (Fav * b)
    OM = Fah * (h / 3)

    # Resultant of the developed weight and lateral earth force
    Rh = Fah
    Rv = W1 + W2 + W3 + W4 + W5 + Fav
    xbar = (RM - OM) / Rv
    ecc = (b / 2) - xbar

    # Maximum soil pressure
    if abs(ecc) <= (b / 6):
        qmax = (Rv / b) * (1 + (ecc / (b / 6)))
    else:
        qmax = (2 * 3) * (Rv / xbar)

    # Safety factors...
    return np.array([
        RM / OM,                # ...against overturning failure
        (D["mu_b"] * Rv) / Rh,  # ...against sliding failure
        D["q_a"] / qmax,        # ...against bearing capacity failure
    ])

## Example 1

### Problem data

In [4]:
D = {
    "H" : 5.0,
    "y_w" : 24.,
    "y_h" : 18.,
    "phi_h" : 30.,
    "c_h" : None,
    "theta" : 15.,
    "phi_b" : None,
    "a_b" : None,
    "mu_b" : 0.45,
    "q_a" : 90.,
    # 
    "b_1_min" : None, "b_1_max" : None,
    "b_2_min" : None, "b_2_max" : None,
    "b_3_min" : None, "b_3_max" : None,
    "b_4_min" : None, "b_4_max" : None,
    "h_c_min" : None, "h_c_max" : None,
    "h_s_min" : None, "h_s_max" : None, # Redundancy
    # 
    "s_o_min" : 2.0,
    "s_s_min" : 1.5,
    "s_b_min" : 3.0,
}
D["b_1_min"] = D["H"] * 0.00
D["b_1_max"] = D["H"] * 0.40
# 
D["b_2_min"] = D["H"] * 0.00    # Redundancy
D["b_2_max"] = D["H"] * 0.40
# 
D["b_3_min"] = D["H"] * 0.00
D["b_3_max"] = D["H"] * 0.40
# 
D["b_4_min"] = D["H"] * 0.05
D["b_4_max"] = D["H"] * 0.20
# 
D["h_c_min"] = D["H"] * 0.05
D["h_c_max"] = D["H"] * 0.20
# 
D["h_s_min"] = D["H"] * 0.00    # Redundancy
D["h_s_max"] = D["H"] * 1.00    # Redundancy

### Objective function, bounds, and constraints

In [5]:
objfun = lambda p : A(p, D)
bounds = spo.Bounds(
    lb = np.array([D["b_1_min"], D["b_2_min"], D["b_3_min"], D["b_4_min"],
                   D["h_s_min"], D["h_c_min"]]),
    ub = np.array([D["b_1_max"], D["b_2_max"], D["b_3_max"], D["b_4_max"],
                   D["h_s_max"], D["h_c_max"]]),
)
sumtoH = spo.LinearConstraint(
    np.array([0., 0., 0., 0., 1., 1.]),
    lb = D["H"], ub = D["H"],
)
SFreqs = spo.NonlinearConstraint(
    lambda p : s_osb(p, D),
    np.array([D["s_o_min"], D["s_s_min"], D["s_b_min"]]),
    np.array([np.inf, np.inf, np.inf]),
)

### Initial estimate

In [6]:
p_init = D["H"] * np.array([0.2, 0.1, 0.2, 0.1, 0.90, 0.10])

### Optimal values

In [7]:
result = spo.minimize(objfun, p_init, bounds = bounds, constraints = [sumtoH, SFreqs])
print(result)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 3.815736261341077
           x: [ 1.987e+00  8.453e-01  2.000e+00  2.527e-01  4.750e+00
                2.500e-01]
         nit: 4
         jac: [ 2.500e-01  2.625e+00  2.500e-01  2.375e+00  5.490e-01
                4.832e+00]
        nfev: 39
        njev: 4
 multipliers: [ 2.598e+00  0.000e+00  8.223e+00  0.000e+00]


### Summary

In [8]:
A_init = A(p_init, D)
s_init = s_osb(p_init, D)

p_optim = result.x
A_optim = A(p_optim, D)
s_optim = s_osb(p_optim, D)

print("Wall proportions [m] (initial -> optimal):")
print(f"{' ' * 3}b_1 = {p_init[0]:.5f} -> {p_optim[0]:.5f}")
print(f"{' ' * 3}b_2 = {p_init[1]:.5f} -> {p_optim[1]:.5f}")
print(f"{' ' * 3}b_3 = {p_init[2]:.5f} -> {p_optim[2]:.5f}")
print(f"{' ' * 3}b_4 = {p_init[3]:.5f} -> {p_optim[3]:.5f}")
print(f"{' ' * 3}h_s = {p_init[4]:.5f} -> {p_optim[4]:.5f}")
print(f"{' ' * 3}h_c = {p_init[5]:.5f} -> {p_optim[5]:.5f}")

print("Cross-sectional area [sq.m.] (initial -> optimal):")
print(f"{' ' * 3}{A_init:.5f} -> {A_optim:.5f}")

print("Safety factors (initial -> optimal):")
print(f"{' ' * 3}FS_o = {s_init[0]:.5f} -> {s_optim[0]:.5f}")
print(f"{' ' * 3}FS_s = {s_init[1]:.5f} -> {s_optim[1]:.5f}")
print(f"{' ' * 3}FS_b = {s_init[2]:.5f} -> {s_optim[2]:.5f}")

Wall proportions [m] (initial -> optimal):
   b_1 = 1.00000 -> 1.98656
   b_2 = 0.50000 -> 0.84533
   b_3 = 1.00000 -> 2.00000
   b_4 = 0.50000 -> 0.25267
   h_s = 4.50000 -> 4.75000
   h_c = 0.50000 -> 0.25000
Cross-sectional area [sq.m.] (initial -> optimal):
   3.50000 -> 3.81574
Safety factors (initial -> optimal):
   FS_o = 2.30927 -> 6.33929
   FS_s = 1.05738 -> 1.50000
   FS_b = 0.72088 -> 4.45622


## Example 2

This is the same as Example 1,
except that the initial estimate is infeasible.

In [9]:
# Stem height (h_s) and footing thickness (h_c) do not add up to the retained height (H)
p_init = D["H"] * np.array([0.2, 0.1, 0.2, 0.1, 1.00, 0.10])

In [10]:
result = spo.minimize(objfun, p_init, bounds = bounds, constraints = [sumtoH, SFreqs])
print(result)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 3.8157362781009643
           x: [ 1.988e+00  8.476e-01  2.000e+00  2.500e-01  4.750e+00
                2.500e-01]
         nit: 4
         jac: [ 2.500e-01  2.625e+00  2.500e-01  2.375e+00  5.488e-01
                4.836e+00]
        nfev: 28
        njev: 4
 multipliers: [ 2.598e+00  0.000e+00  8.223e+00  0.000e+00]


In [11]:
A_init = A(p_init, D)
s_init = s_osb(p_init, D)

p_optim = result.x
A_optim = A(p_optim, D)
s_optim = s_osb(p_optim, D)

print("Wall proportions [m] (initial -> optimal):")
print(f"{' ' * 3}b_1 = {p_init[0]:.5f} -> {p_optim[0]:.5f}")
print(f"{' ' * 3}b_2 = {p_init[1]:.5f} -> {p_optim[1]:.5f}")
print(f"{' ' * 3}b_3 = {p_init[2]:.5f} -> {p_optim[2]:.5f}")
print(f"{' ' * 3}b_4 = {p_init[3]:.5f} -> {p_optim[3]:.5f}")
print(f"{' ' * 3}h_s = {p_init[4]:.5f} -> {p_optim[4]:.5f}")
print(f"{' ' * 3}h_c = {p_init[5]:.5f} -> {p_optim[5]:.5f}")

print("Cross-sectional area [sq.m.] (initial -> optimal):")
print(f"{' ' * 3}{A_init:.5f} -> {A_optim:.5f}")

print("Safety factors (initial -> optimal):")
print(f"{' ' * 3}FS_o = {s_init[0]:.5f} -> {s_optim[0]:.5f}")
print(f"{' ' * 3}FS_s = {s_init[1]:.5f} -> {s_optim[1]:.5f}")
print(f"{' ' * 3}FS_b = {s_init[2]:.5f} -> {s_optim[2]:.5f}")

Wall proportions [m] (initial -> optimal):
   b_1 = 1.00000 -> 1.98843
   b_2 = 0.50000 -> 0.84757
   b_3 = 1.00000 -> 2.00000
   b_4 = 0.50000 -> 0.25000
   h_s = 5.00000 -> 4.75000
   h_c = 0.50000 -> 0.25000
Cross-sectional area [sq.m.] (initial -> optimal):
   3.75000 -> 3.81574
Safety factors (initial -> optimal):
   FS_o = 1.95464 -> 6.34621
   FS_s = 0.97202 -> 1.50000
   FS_b = 0.55107 -> 4.48214
